In [ ]:
# @title Complete Forestry Training Pipeline (Single Cell - Colab)
# Runs: Setup → Download ForestNet + LEVIR-CD → Prepare → Train YOLO → Train ChangeFormer → Evaluate → Verify
# Expected time: ~3-4 hours on T4

import os, sys, subprocess, json, time
from pathlib import Path

# ==== CONFIG ====
REPO_URL = "https://github.com/YOUR_ORG/impact-platform.git"  # <-- CHANGE THIS
REPO_DIR = "/content/drive/MyDrive/impact-platform"
BATCH_SIZE = 8  # reduce if OOM
EPOCHS_YOLO = 50
EPOCHS_CHANGE = 200

def run(cmd, desc):
    print(f"\n{'='*60}")
    print(f"▶ {desc}")
    print(f"{'='*60}")
    start = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.time() - start
    if result.returncode != 0:
        print(f"❌ FAILED after {elapsed:.0f}s")
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"{desc} failed")
    print(f"✅ DONE in {elapsed:.0f}s")
    return result.stdout

# ==== 1. SETUP ====
print("🔧 Setting up repository...")
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists(REPO_DIR):
    run(f"git clone {REPO_URL} {REPO_DIR}", "Clone repo")
else:
    run(f"git -C {REPO_DIR} pull", "Pull latest")
os.chdir(f"{REPO_DIR}/apps/ml-service")
print(f"Working in: {os.getcwd()}")

# ==== 2. INSTALL DEPS ====
run("pip install -q -r training/requirements_colab.txt", "Install requirements")

# ==== 3. DOWNLOAD FORESTNET ====
print("📥 Downloading ForestNet (GEO-Bench, ~1.2 GB)...")
run("python -m training.data.download_datasets download_forestnet", "Download ForestNet")

# ==== 4. DOWNLOAD LEVIR-CD ====
print("📥 Downloading LEVIR-CD (train/val/test, ~2.5 GB)...")
run("python -m training.data.download_datasets download_levir_cd", "Download LEVIR-CD")

# ==== 5. PREPARE YOLO FORMAT (forestry) ====
run("python -m training.data.prepare_yolo forestry", "Prepare YOLO format (forestry)")

# ==== 6. PREPARE LEVIR-CD FOR CHANGEFORMER ====
run("python -m training.data.prepare_yolo levir_cd", "Prepare LEVIR-CD for ChangeFormer")

# ==== 7. TRAIN YOLOv8n SAPLING ====
print(f"🏋️ Training YOLOv8n sapling for {EPOCHS_YOLO} epochs (batch={BATCH_SIZE})...")
os.environ["TRAINING_BATCH_SIZE"] = str(BATCH_SIZE)
run(f"python -m training.scripts.train_yolo forestry", f"Train YOLOv8n sapling ({EPOCHS_YOLO} epochs)")

# ==== 8. TRAIN CHANGEFORMER (forestry) ====
print(f"🏋️ Training ChangeFormer for {EPOCHS_CHANGE} epochs...")
run(f"python -m training.scripts.train_change forestry", f"Train ChangeFormer ({EPOCHS_CHANGE} epochs)")

# ==== 9. EVALUATE ====
run(f"python -m training.scripts.evaluate forestry --output eval/forestry_eval_v1.0.json", "Evaluate")

# ==== 10. VERIFY WEIGHTS ====
print("\n" + "="*60)
print("📦 VERIFYING WEIGHTS")
print("="*60)
weights_dir = Path("weights")
for w in ["sapling_yolov8n.pt", "changeformer.pt", "yolov8n.pt"]:
    p = weights_dir / w
    if p.exists():
        size = p.stat().st_size / 1e6
        print(f"  ✅ {w}: {size:.1f} MB")
    else:
        print(f"  ❌ {w}: MISSING")

# ==== 11. QUICK INFERENCE TEST ====
print("\n🧪 QUICK INFERENCE TEST")
try:
    from ultralytics import YOLO
    model = YOLO("weights/sapling_yolov8n.pt")
    print("✅ YOLO model loads successfully")
except Exception as e:
    print(f"❌ YOLO load failed: {e}")

try:
    import torch
    from training.scripts.train_change import ChangeFormer
    model = ChangeFormer()
    model.load_state_dict(torch.load("weights/changeformer.pt"))
    model.eval()
    print("✅ ChangeFormer loads successfully")
except Exception as e:
    print(f"❌ ChangeFormer load failed: {e}")

print("\n🎉 FORESTRY PIPELINE COMPLETE!")
print("Next: python -m training.scripts.promote_model forestry --version v1.0")
